In [47]:
import numpy as np
from scipy.stats import ttest_ind, chi2_contingency, f_oneway
import pandas as pd
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import f_regression, SelectKBest
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

### What is Hypothesis Testing?

**Hypothesis Testing** is a statistical procedure for determining whether there is enough evidence to reject a default assumption (the **"null hypothesis"**) in favor of an alternative.



| Term | Meaning | Example |
|------|---------|---------|
| **Null Hypothesis ($H_0$)** | Default assumption | "The new model is no better than the old model" |
| **Alternative Hypothesis ($H_1$)** | What you want to prove | "The new model is better than the old model" |
| **p-value** | Probability of observing data if $H_0$ is true | p < 0.05 → reject $H_0$ |
| **Significance Level ($\alpha$)** | Threshold for rejecting $H_0$ | $\alpha = 0.05$ (95% confidence) |

> **The Golden Rule:** In ML, we use hypothesis testing for **A/B testing**, **feature selection**, and validating that our model actually learned something meaningful (not just random noise).

The Most Common Mistake:

❌ **"The p-value is the probability that the null hypothesis is true."**

This is **WRONG!** The p-value does **not** tell you how likely the null hypothesis is.


The Correct Definition:

✅ **"The p-value is the probability of observing data as extreme as we did, assuming the null hypothesis is true."**


| Wrong Interpretation | Correct Interpretation |
|----------------------|----------------------|
| "There's a 3% chance the null is true" | "If the null were true, there's a 3% chance we'd see this data" |
| p = P(H₀ \| Data) ❌ | p = P(Data \| H₀) ✅ |


In [11]:
# T-Tests (For Comparing Two Groups)

# Example: A/B test for website click-through rate
# Version A: 1000 users, 50 clicks
# Version B: 1000 users, 65 clicks

# H₀: μ_A = μ_B  (population means are equal)
# H₁: μ_A ≠ μ_B  (population means are different)

# click data
np.random.seed(42)
A_clicks = np.random.binomial(1, 0.05, 1000)  # 5% CTR
B_clicks = np.random.binomial(1, 0.065, 1000)  # 6.5% CTR

# T-test
t_stat, p_value = ttest_ind(A_clicks, B_clicks)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Version A CTR: {A_clicks.mean():.3f}")
print(f"Version B CTR: {B_clicks.mean():.3f}")

alpha = 0.05  # 5% significance level (CI = 95%)

if p_value < alpha:
    print("✅ Statistically significant difference")
    if B_clicks.mean() > A_clicks.mean():
        print("✅ Version B has higher CTR → Version B is better!")
    else:
        print("✅ Version A has higher CTR → Version A is better!")
else:
    print("❌ Not statistically significant")

T-statistic: -2.6396
P-value: 0.0084
Version A CTR: 0.046
Version B CTR: 0.074
✅ Statistically significant difference
✅ Version B has higher CTR → Version B is better!


In [19]:
# Chi-Square Test (For Categorical Features)

# Example: Does gender affect whether someone buys a product?
# Observed data (contingency table)
data = pd.DataFrame({
    'Male': [120, 80],
    'Female': [100, 100]
}, index=['Bought', 'Did not buy'])

print("Contingency Table:")
print(data)

# Chi-square test
chi2, p_value, dof, expected = chi2_contingency(data)

print('='*50)
print("""
CHI-SQUARE TEST HYPOTHESES:                                                                                          
NULL HYPOTHESIS (H₀):                                    
Gender and purchase behavior are INDEPENDENT             
(There is NO association between gender and buying)      
      
ALTERNATIVE HYPOTHESIS (H₁):                             
Gender and purchase behavior are DEPENDENT               
(There IS an association between gender and buying)      
""")
print('='*50)

print(f"\nChi-square statistic: {chi2:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("✅ Gender is significantly associated with purchase behavior.")
else:
    print("❌ No significant association between gender and purchase behavior.")

print('='*50)
print("\nExpected frequencies (if no association):")
print(pd.DataFrame(expected, index=data.index, columns=data.columns).round(1))

Contingency Table:
             Male  Female
Bought        120     100
Did not buy    80     100

CHI-SQUARE TEST HYPOTHESES:                                                                                          
NULL HYPOTHESIS (H₀):                                    
Gender and purchase behavior are INDEPENDENT             
(There is NO association between gender and buying)      
      
ALTERNATIVE HYPOTHESIS (H₁):                             
Gender and purchase behavior are DEPENDENT               
(There IS an association between gender and buying)      


Chi-square statistic: 3.6465
P-value: 0.0562
❌ No significant association between gender and purchase behavior.

Expected frequencies (if no association):
              Male  Female
Bought       110.0   110.0
Did not buy   90.0    90.0


In [35]:
# ANOVA (For Multiple Groups)

# Example: Does the marketing channel affect sales?
# Group 1: Social Media
# Group 2: Email
# Group 3: Direct Mail

social_media = np.random.normal(100, 15, 30) # mean=100, std=30, sample_size=30
email = np.random.normal(110, 12, 30)
direct_mail = np.random.normal(95, 18, 30)

f_stat, p_value = f_oneway(social_media, email, direct_mail)

print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("✅ At least one channel is significantly different from the others.")
    print("(But which one? Need post-hoc testing for that.)")
else:
    print("❌ No significant difference between channels.")


print("""
================================================================
POST-HOC TESTING?                                      
================================================================

ANOVA Result: "At least one group is different"            
                                                           
But we need to know:                                       
    • Is Social Media different from Email?                    
    • Is Social Media different from Direct Mail?              
    • Is Email different from Direct Mail?                                                                                                                                       
""")

# data for Tukey's test
all_data = np.concatenate([social_media, email, direct_mail])
groups = ['Social Media']*30 + ['Email']*30 + ['Direct Mail']*30

# Tukey's HSD test
tukey_result = pairwise_tukeyhsd(all_data, groups, alpha=0.05)


print(tukey_result)


F-statistic: 6.9937
P-value: 0.0015
✅ At least one channel is significantly different from the others.
(But which one? Need post-hoc testing for that.)

POST-HOC TESTING?                                      

ANOVA Result: "At least one group is different"            
                                                           
But we need to know:                                       
    • Is Social Media different from Email?                    
    • Is Social Media different from Direct Mail?              
    • Is Email different from Direct Mail?                                                                                                                                       

      Multiple Comparison of Means - Tukey HSD, FWER=0.05       
   group1      group2    meandiff p-adj   lower    upper  reject
----------------------------------------------------------------
Direct Mail        Email  14.6912  0.001   5.3059 24.0765   True
Direct Mail Social Media   8.1494  0.102  -

| Comparison | Significant? | Meaning |
|------------|--------------|---------|
| **Direct Mail vs Email** | ✅ Yes | The two channels perform differently |
| **Email vs Social Media** | ✅ Yes | The two channels perform differently |
| **Direct Mail vs Social Media** | ❌ No | No statistically significant difference |



In [36]:
# ANOVA (For Multiple Groups)

np.random.seed(42)

# A/B/C Test Data
version_A = np.random.normal(100, 15, 30)   # Original version
version_B = np.random.normal(110, 12, 30)   # New design
version_C = np.random.normal(105, 14, 30)   # Another variant

print("="*60)
print("A/B/C TEST RESULTS")
print("="*60)

# 1. ANOVA - "Is there any difference?"
f_stat, p_value = f_oneway(version_A, version_B, version_C)

print(f"ANOVA Results:")
print(f"  F-statistic: {f_stat:.4f}")
print(f"  P-value: {p_value:.4f}")

if p_value < 0.05:
    print("  ✅ Significant difference found among versions!")
    print("  → Need for post-hoc tests.")
else:
    print("  ❌ No significant difference found.")
    print("  → Stop here. No need for post-hoc tests.")

# 2. Post-hoc tests - "Which versions are different?"
if p_value < 0.05:
    print("\n" + "-"*60)
    print("POST-HOC TEST (Tukey HSD)")
    print("-"*60)
    
    # all data
    all_data = np.concatenate([version_A, version_B, version_C])
    groups = ['A']*30 + ['B']*30 + ['C']*30
    
    # Tukey's HSD
    tukey = pairwise_tukeyhsd(all_data, groups, alpha=0.05)
    print(tukey)
    
    # Calculate means
    means = {
        'A': version_A.mean(),
        'B': version_B.mean(),
        'C': version_C.mean()
    }
    
    print("\n" + "-"*60)
    print("MEANS:")
    print("-"*60)
    for version, mean in means.items():
        print(f"  Version {version}: {mean:.2f}")
    
    print("\n" + "-"*60)
    print("RANKING:")
    print("-"*60)
    sorted_versions = sorted(means.items(), key=lambda x: x[1], reverse=True)
    for i, (version, mean) in enumerate(sorted_versions, 1):
        medal = ['🥇', '🥈', '🥉'][i-1] if i <= 3 else ''
        print(f"  {medal} Version {version}: {mean:.2f}")

A/B/C TEST RESULTS
ANOVA Results:
  F-statistic: 6.1386
  P-value: 0.0032
  ✅ Significant difference found among versions!

------------------------------------------------------------
POST-HOC TEST (Tukey HSD)
------------------------------------------------------------
 Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower    upper  reject
-----------------------------------------------------
     A      B  11.3683 0.0028   3.4203 19.3162   True
     A      C   8.0026 0.0481   0.0546 15.9506   True
     B      C  -3.3657 0.5727 -11.3136  4.5823  False
-----------------------------------------------------

------------------------------------------------------------
MEANS:
------------------------------------------------------------
  Version A: 97.18
  Version B: 108.55
  Version C: 105.18

------------------------------------------------------------
RANKING:
------------------------------------------------------------
  🥇 Version B: 108.55
  🥈 Ver

In [39]:
print("\n" + "="*60)
print("RESULTS ANALYSIS")
print("="*60)

means = {'A': 97.18, 'B': 108.55, 'C': 105.18}
tukey_results = [
    {'pair': 'A vs B', 'diff': 11.37, 'p': 0.0028, 'significant': True},
    {'pair': 'A vs C', 'diff': 8.00, 'p': 0.0481, 'significant': True},
    {'pair': 'B vs C', 'diff': -3.37, 'p': 0.5727, 'significant': False}
]

print("\n📊 MEANS ONLY:")
print(f"  🥇 B: 108.55")
print(f"  🥈 C: 105.18")
print(f"  🥉 A: 97.18")
print("\n  Based on means alone: B is best, C is second, A is worst")

print("\n📊 POST-HOC TEST (Tukey HSD):")
for result in tukey_results:
    status = "✅ SIGNIFICANT" if result['significant'] else "❌ NOT SIGNIFICANT"
    print(f"  {result['pair']}: diff={result['diff']:.2f}, p={result['p']:.4f} → {status}")

print("\n🔍 WHAT THIS TELLS US:")
print("  ✅ A vs B: B is significantly better than A")
print("  ✅ A vs C: C is significantly better than A")
print("  ❌ B vs C: NO significant difference between B and C")

print("\n💡 BUSINESS INSIGHT:")
print("  • Both B and C are better than A")
print("  • But B and C are NOT significantly different")
print("  • If B is more expensive, C might be the better choice!")
print("  • Cannot conclude B is better than C, despite B having higher mean")


RESULTS ANALYSIS

📊 MEANS ONLY:
  🥇 B: 108.55
  🥈 C: 105.18
  🥉 A: 97.18

  Based on means alone: B is best, C is second, A is worst

📊 POST-HOC TEST (Tukey HSD):
  A vs B: diff=11.37, p=0.0028 → ✅ SIGNIFICANT
  A vs C: diff=8.00, p=0.0481 → ✅ SIGNIFICANT
  B vs C: diff=-3.37, p=0.5727 → ❌ NOT SIGNIFICANT

🔍 WHAT THIS TELLS US:
  ✅ A vs B: B is significantly better than A
  ✅ A vs C: C is significantly better than A
  ❌ B vs C: NO significant difference between B and C

💡 BUSINESS INSIGHT:
  • Both B and C are better than A
  • But B and C are NOT significantly different
  • If B is more expensive, C might be the better choice!
  • Cannot conclude B is better than C, despite B having higher mean


In [45]:
# Hypothesis Testing in Feature Selection

np.random.seed(42)
X, y = make_regression(n_samples=200, n_features=10, n_informative=5, noise=0.1)

# split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# F-test for feature significance
f_scores_train, p_values_train = f_regression(X_train, y_train)

feature_df = pd.DataFrame({
    'Feature': [f'Feature_{i}' for i in range(10)],
    'F-score': f_scores,
    'P-value': p_values
})
feature_df = feature_df.sort_values('P-value')

significant_features = feature_df[feature_df['P-value'] < 0.05]['Feature'].tolist()
print(f"\nSignificant Features (p < 0.05): {significant_features}")

# train model with ALL features
model_all = LinearRegression()
model_all.fit(X_train, y_train)
y_pred_all = model_all.predict(X_test)
r2_all = r2_score(y_test, y_pred_all)

# train model with SELECTED features
feature_indices = [int(f.split('_')[1]) for f in significant_features]
X_train_selected = X_train[:, feature_indices]
X_test_selected = X_test[:, feature_indices]

model_selected = LinearRegression()
model_selected.fit(X_train_selected, y_train)
y_pred_selected = model_selected.predict(X_test_selected)
r2_selected = r2_score(y_test, y_pred_selected)

print("\nModel Performance Comparison:")
print(f"  All features: R² = {r2_all:.4f}")
print(f"  Selected features: R² = {r2_selected:.4f}")


Significant Features (p < 0.05): ['Feature_3', 'Feature_5', 'Feature_7']

Model Performance Comparison:
  All features: R² = 1.0000
  Selected features: R² = 0.9627


In [52]:
# feature selection
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_selection', SelectKBest(f_regression, k=5)),  # Select top 5 features
    ('model', LinearRegression())
])

# train on training data only
pipeline.fit(X_train, y_train)

# evaluate on test data
y_pred = pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)

print(f"Test R²: {r2:.4f}")

# selected features
selected_mask = pipeline.named_steps['feature_selection'].get_support()
selected_features = [f'Feature_{i}' for i, selected in enumerate(selected_mask) if selected]
print(f"Selected features: {selected_features}")

Test R²: 0.9762
Selected features: ['Feature_1', 'Feature_3', 'Feature_5', 'Feature_6', 'Feature_7']
